# SoundStream demo

This notebook downloads the checkpoint, reconstructs audio from a user-provided URL, and reports the compression ratio.

Clone the repository and install the required packages.

In [ ]:
!git clone https://github.com/daminovkamil/soundstream-codec.git
%cd soundstream-codec
!pip install -q -r requirements.txt huggingface_hub

Prepare a local directory for downloaded artifacts and define the input audio URL.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse

import requests
from huggingface_hub import hf_hub_download

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

# Set a direct link to an audio file here.
AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

Download the pretrained checkpoint from Hugging Face.

In [ ]:
checkpoint_path = Path(
    hf_hub_download(
        repo_id="daminovkamil/soundstream-codec",
        filename="best-40000-161.865.ckpt",
        local_dir=artifacts_dir,
    )
)

print(f"Checkpoint: {checkpoint_path}")

Download the input audio from the provided URL.

In [ ]:
filename = Path(urlparse(AUDIO_URL).path).name or "input.wav"
audio_path = artifacts_dir / filename

!curl -fsSL -o "{audio_path}" "{AUDIO_URL}"
print(f"Audio: {audio_path}")

Import the model and define the codec configuration.

In [ ]:
import torch
import torchaudio
from src.model import SoundStream

SAMPLE_RATE = 16_000
STRIDES = [2, 4, 5, 5]
CHANNELS = 32
EMBEDDING_DIM = 128
NUM_QUANTIZERS = 8
CODEBOOK_SIZE = 1024

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Load the checkpoint, preprocess the audio to 16 kHz mono, and run reconstruction.

In [ ]:
codec = SoundStream.from_pretrained(
    checkpoint_path,
    CHANNELS,
    EMBEDDING_DIM,
    NUM_QUANTIZERS,
    CODEBOOK_SIZE,
    STRIDES,
    map_location=device,
).to(device)

# Load the saved audio file and convert it to 16 kHz mono.
waveform, sample_rate = torchaudio.load(audio_path)
if waveform.size(0) > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
if sample_rate != SAMPLE_RATE:
    waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

model_input = waveform.unsqueeze(0).clamp(-1.0, 1.0).to(device)
with torch.no_grad():
    encoded = codec.encode(model_input)
    indices = codec.quantize(encoded)
    quantized = codec.unquantize(indices)
    reconstructed = codec.decode(quantized, length=model_input.size(-1))

reconstructed = reconstructed.squeeze(0).cpu()
reconstructed_path = artifacts_dir / "reconstructed.wav"
torchaudio.save(str(reconstructed_path), reconstructed, SAMPLE_RATE)

print(f"Saved reconstruction to {reconstructed_path}")

Compute the compression ratio from the discrete codes.

In [ ]:
# Estimate bitrate and compression ratio for 16-bit mono PCM.
import math

bits_per_code = math.ceil(math.log2(CODEBOOK_SIZE))
compressed_bits = indices.numel() * bits_per_code
original_bits = model_input.size(-1) * 16

compressed_bitrate = compressed_bits * SAMPLE_RATE / model_input.size(-1)
original_bitrate = SAMPLE_RATE * 16
compression_ratio = original_bits / compressed_bits

print(f"Original bitrate: {original_bitrate / 1000:.1f} kbps")
print(f"Compressed bitrate: {compressed_bitrate / 1000:.1f} kbps")
print(f"Compression ratio: {compression_ratio:.1f}x")

Listen to the original audio and the reconstructed audio.

In [ ]:
# Play the original and reconstructed waveforms.
from IPython.display import Audio, display

print("Original")
display(Audio(waveform.squeeze(0).numpy(), rate=SAMPLE_RATE))

print("Reconstructed")
display(Audio(reconstructed.squeeze(0).numpy(), rate=SAMPLE_RATE))